# Olist Commerce Intelligence: Analytics + Late Delivery Prediction

This end-to-end case study connects nine relational tables from the Brazilian Olist marketplace, analyzes commercial and delivery performance, segments customers, and builds a leakage-aware model to identify orders at risk of arriving after the promised date.

## Business questions

1. Where do revenue and order volume come from?
2. How is delivery reliability associated with customer ratings?
3. Which customer groups appear in an RFM view?
4. Can checkout-time information rank late-delivery risk?

The predictive section uses a chronological holdout and excludes delivery timestamps, reviews, and every other post-purchase outcome from the feature set.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

INPUT = Path('/kaggle/input')

def find_file(name):
    matches = list(INPUT.rglob(name))
    if not matches:
        raise FileNotFoundError(f'Attach the Olist dataset; missing {name}')
    return matches[0]

def read(name):
    return pd.read_csv(find_file(name))

customers = read('olist_customers_dataset.csv')
geo = read('olist_geolocation_dataset.csv')
items = read('olist_order_items_dataset.csv')
payments = read('olist_order_payments_dataset.csv')
reviews = read('olist_order_reviews_dataset.csv')
orders = read('olist_orders_dataset.csv')
products = read('olist_products_dataset.csv')
sellers = read('olist_sellers_dataset.csv')
translation = read('product_category_name_translation.csv')

pd.DataFrame({
    'table': ['customers','geolocation','items','payments','reviews','orders','products','sellers','translation'],
    'rows': [len(customers),len(geo),len(items),len(payments),len(reviews),len(orders),len(products),len(sellers),len(translation)]
})

## Data preparation and feature engineering

In [ ]:
date_cols = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

geo_zip = geo.groupby('geolocation_zip_code_prefix', as_index=False).agg(
    lat=('geolocation_lat','median'), lng=('geolocation_lng','median'))
customers = customers.merge(geo_zip.rename(columns={'geolocation_zip_code_prefix':'customer_zip_code_prefix','lat':'customer_lat','lng':'customer_lng'}), on='customer_zip_code_prefix', how='left')
sellers = sellers.merge(geo_zip.rename(columns={'geolocation_zip_code_prefix':'seller_zip_code_prefix','lat':'seller_lat','lng':'seller_lng'}), on='seller_zip_code_prefix', how='left')

products = products.merge(translation, on='product_category_name', how='left')
products['category'] = products.product_category_name_english.fillna(products.product_category_name)
products['volume_cm3'] = products.product_length_cm * products.product_height_cm * products.product_width_cm

detail = (items
    .merge(products[['product_id','category','product_weight_g','volume_cm3']], on='product_id', how='left')
    .merge(sellers[['seller_id','seller_state','seller_lat','seller_lng']], on='seller_id', how='left')
    .merge(orders[['order_id','customer_id']].merge(customers[['customer_id','customer_lat','customer_lng']], on='customer_id', how='left'), on='order_id', how='left'))

def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 2 * 6371.0088 * np.arcsin(np.sqrt(a))

detail['distance_km'] = haversine(detail.customer_lat, detail.customer_lng, detail.seller_lat, detail.seller_lng)
agg = detail.groupby('order_id', as_index=False).agg(
    item_count=('order_item_id','count'), unique_product_count=('product_id','nunique'), seller_count=('seller_id','nunique'),
    total_price=('price','sum'), total_freight=('freight_value','sum'), avg_product_weight_g=('product_weight_g','mean'),
    avg_product_volume_cm3=('volume_cm3','mean'), avg_distance_km=('distance_km','mean'))

def primary_value(frame, col, out):
    return (frame.dropna(subset=[col]).groupby(['order_id',col]).size().reset_index(name='n')
            .sort_values(['order_id','n'], ascending=[True,False]).drop_duplicates('order_id')
            [['order_id',col]].rename(columns={col:out}))

agg = agg.merge(primary_value(detail,'category','primary_category'), on='order_id', how='left')
agg = agg.merge(primary_value(detail,'seller_state','primary_seller_state'), on='order_id', how='left')
pay = payments.groupby('order_id', as_index=False).agg(total_payment=('payment_value','sum'), payment_installments=('payment_installments','max'))
main_pay = payments.sort_values(['order_id','payment_value'], ascending=[True,False]).drop_duplicates('order_id')[['order_id','payment_type']].rename(columns={'payment_type':'primary_payment_type'})
pay = pay.merge(main_pay, on='order_id', how='left')
review = reviews.groupby('order_id', as_index=False).agg(review_score=('review_score','mean'))

df = orders.merge(customers, on='customer_id', how='left').merge(agg, on='order_id', how='left').merge(pay, on='order_id', how='left').merge(review, on='order_id', how='left')
df['purchase_year_month'] = df.order_purchase_timestamp.dt.to_period('M').astype(str)
df['purchase_month'] = df.order_purchase_timestamp.dt.month
df['purchase_weekday'] = df.order_purchase_timestamp.dt.dayofweek
df['purchase_hour'] = df.order_purchase_timestamp.dt.hour
df['estimated_days'] = (df.order_estimated_delivery_date-df.order_purchase_timestamp).dt.total_seconds()/86400
df['delivery_days'] = (df.order_delivered_customer_date-df.order_purchase_timestamp).dt.total_seconds()/86400
df['late_days'] = (df.order_delivered_customer_date-df.order_estimated_delivery_date).dt.total_seconds()/86400
df['is_late'] = np.where(df.order_delivered_customer_date.notna(), (df.late_days>0).astype(int), np.nan)
delivered = df.query("order_status == 'delivered'").copy()
print(f'{df.order_id.nunique():,} orders; {len(delivered):,} delivered; late rate {delivered.is_late.mean():.1%}')

## Executive analytics

In [ ]:
kpis = pd.Series({
    'Orders': df.order_id.nunique(), 'Delivered orders': delivered.order_id.nunique(),
    'Delivered revenue (BRL)': delivered.total_payment.sum(), 'Average order value (BRL)': delivered.total_payment.mean(),
    'Late-delivery rate': delivered.is_late.mean(), 'Average review': delivered.review_score.mean()})
display(kpis.to_frame('value'))

monthly = delivered.groupby('purchase_year_month', as_index=False).agg(revenue=('total_payment','sum'), orders=('order_id','nunique'), late_rate=('is_late','mean'))
px.line(monthly, x='purchase_year_month', y='revenue', markers=True, title='Monthly delivered revenue').show()

category = delivered.groupby('primary_category', as_index=False).agg(orders=('order_id','nunique'), revenue=('total_payment','sum'), late_rate=('is_late','mean'), review=('review_score','mean'))
px.bar(category.nlargest(12,'revenue').sort_values('revenue'), x='revenue', y='primary_category', orientation='h', title='Top categories by revenue').show()

review_delivery = delivered.assign(delivery_result=np.where(delivered.is_late.eq(1),'Late','On time')).groupby('delivery_result', as_index=False).agg(orders=('order_id','nunique'), average_review=('review_score','mean'))
display(review_delivery)
px.bar(review_delivery, x='delivery_result', y='average_review', color='delivery_result', range_y=[0,5], title='Average review by delivery result').show()

## RFM customer segmentation
Because repeat purchase is rare in this observation window, these segments are descriptive and should not be presented as validated campaign lift.

In [ ]:
reference = delivered.order_purchase_timestamp.max().normalize() + pd.offsets.Day(1)
rfm = delivered.groupby('customer_unique_id', as_index=False).agg(
    recency=('order_purchase_timestamp', lambda x:(reference-x.max()).days), frequency=('order_id','nunique'), monetary=('total_payment','sum'))
rfm['r'] = pd.cut(rfm.recency.rank(method='first',pct=True), [0,.25,.5,.75,1], labels=[4,3,2,1], include_lowest=True).astype(int)
rfm['f'] = pd.cut(rfm.frequency.rank(method='first',pct=True), [0,.25,.5,.75,1], labels=[1,2,3,4], include_lowest=True).astype(int)
def segment(x):
    if x.r>=4 and x.f>=3: return 'Champions'
    if x.r>=3 and x.f>=3: return 'Loyal'
    if x.r>=3: return 'Recent'
    if x.f>=3: return 'At risk'
    return 'Hibernating'
rfm['segment'] = rfm.apply(segment, axis=1)
summary = rfm.groupby('segment', as_index=False).agg(customers=('customer_unique_id','nunique'), revenue=('monetary','sum'), avg_recency=('recency','mean'))
display(summary.sort_values('revenue',ascending=False))
px.treemap(summary, path=['segment'], values='revenue', title='Revenue represented by RFM segment').show()
print(f'Repeat-customer rate: {(rfm.frequency>1).mean():.1%}')

## Late-delivery prediction

The oldest 80% of delivered orders form the training period; the newest 20% form the untouched test period. A validation slice inside the training period selects the model and threshold. PR-AUC is emphasized because late deliveries are uncommon.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num = ['item_count','unique_product_count','seller_count','total_price','total_freight','avg_product_weight_g','avg_product_volume_cm3','avg_distance_km','payment_installments','estimated_days','purchase_month','purchase_weekday','purchase_hour']
cat = ['customer_state','primary_category','primary_seller_state','primary_payment_type']
features = num + cat
model_data = delivered.dropna(subset=['is_late','order_purchase_timestamp']).sort_values('order_purchase_timestamp').reset_index(drop=True)
model_data['is_late'] = model_data.is_late.astype(int)
cut = int(len(model_data)*.8); train, test = model_data.iloc[:cut], model_data.iloc[cut:]
vcut = int(len(train)*.8); fit, valid = train.iloc[:vcut], train.iloc[vcut:]

def prep():
    return ColumnTransformer([
        ('num', Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]), num),
        ('cat', Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore',min_frequency=20))]), cat)])

models = {
    'Dummy': DummyClassifier(strategy='prior'),
    'Logistic regression': LogisticRegression(max_iter=1200,class_weight='balanced'),
    'Random forest': RandomForestClassifier(n_estimators=150,min_samples_leaf=8,max_features='sqrt',class_weight='balanced_subsample',random_state=42,n_jobs=-1)}
validation = {}
for name, estimator in models.items():
    pipe = Pipeline([('prep',prep()),('model',estimator)]).fit(fit[features],fit.is_late)
    p = pipe.predict_proba(valid[features])[:,1]
    thresholds = np.linspace(.05,.8,76)
    threshold = .5 if name=='Dummy' else thresholds[np.argmax([f1_score(valid.is_late,p>=t) for t in thresholds])]
    validation[name] = {'PR-AUC':average_precision_score(valid.is_late,p),'ROC-AUC':roc_auc_score(valid.is_late,p),'threshold':threshold}
display(pd.DataFrame(validation).T)

winner = max([x for x in models if x!='Dummy'], key=lambda x:validation[x]['PR-AUC'])
final = Pipeline([('prep',prep()),('model',models[winner])]).fit(train[features],train.is_late)
p = final.predict_proba(test[features])[:,1]; t = validation[winner]['threshold']; pred = p>=t
result = pd.Series({'Selected model':winner,'Test late rate':test.is_late.mean(),'PR-AUC':average_precision_score(test.is_late,p),'ROC-AUC':roc_auc_score(test.is_late,p),'Precision':precision_score(test.is_late,pred),'Recall':recall_score(test.is_late,pred),'F1':f1_score(test.is_late,pred),'Threshold':t})
display(result.to_frame('value'))

## Decision and limitations

Use the score to rank orders for proactive monitoring, then test an intervention such as an operational alert or customer notification. The association between lateness and reviews is not causal. Model coefficients are not causal effects.

The data covers an anonymized Brazilian marketplace from 2016–2018 and omits inventory, carrier identity, margin, marketing exposure, and intervention outcomes. Performance must be revalidated before use in another company or time period.